# E3：Focal Tversky Loss 对照实验

已内嵌代码、配置、冻结切分和测试，无需从 GitHub 下载源码。
先运行环境准备与验证两格，核对测试通过后再启动训练。
正式训练和最终评估默认关闭，Run All 不会直接训练。

## 运行前设置
- GPU：选择 T4，与团队基线尽量一致。
- 挂载数据：ozguraslank/brain-stroke-ct-dataset。
- 沿用基线离线依赖：amitmalakarjoy/torch-2100-cu128、koumeimaki/vllm019-cp311-cu128-wheelhouse。
- 配置固定为 E3；基础红色掩码、无增强、种子 42、256×256。
- alpha=0.3 惩罚 FP，beta=0.7 惩罚 FN，gamma=0.75 为直接指数。
- 环境格按仓库安装挂载的 PyTorch 2.10.0+cu128 和 NVIDIA wheels。
- 保持 Internet off；已内嵌从官方 PyTorch/PyPI 固定的哈希，安装前校验离线包。
- 大型安装包使用已挂载文件，不重新下载。校验或 CUDA 检查失败则停止。
- 正式训练使用与仓库一致的 15 轮上限和连续 4 轮未提升早停规则。
- 每次初始化使用独立目录，已有实验结果不会被删除。

In [ ]:
from pathlib import Path
import base64
import io
import os
import shutil
import subprocess
import sys
import tarfile

import uuid
repo = Path('/kaggle/working') / ('brain-stroke-e3-' + uuid.uuid4().hex[:12])
repo.mkdir(parents=True)

payload = '...'
with tarfile.open(fileobj=io.BytesIO(base64.b64decode(payload)), mode='r:gz') as archive:
    archive.extractall(repo, filter='data')

candidates = [
    path for path in Path('/kaggle/input').rglob('Brain_Stroke_CT_Dataset')
    if all((path / name).is_dir() for name in ('Bleeding', 'Ischemia', 'Normal', 'External_Test'))
]
if not candidates:
    raise FileNotFoundError('Brain_Stroke_CT_Dataset was not mounted')
data_root = candidates[0]
os.environ['BSC_DATA_ROOT'] = str(data_root)
os.environ['PYTHONUNBUFFERED'] = '1'
os.environ['MPLBACKEND'] = 'Agg'
os.environ['REQUIRE_CUDA'] = '1'
config_path = 'configs/experiments/E3_focal_tversky.yaml'
import torch
print('torch:', torch.__version__)
print('torch_cuda:', torch.version.cuda)
print('cuda_device_count:', torch.cuda.device_count())
print('cuda_available:', torch.cuda.is_available())
print('CUDA_VISIBLE_DEVICES:', os.environ.get('CUDA_VISIBLE_DEVICES'))
try:
    nvidia_output = subprocess.run(
        ['nvidia-smi'], check=False, capture_output=True, text=True, timeout=15
    )
    print('nvidia-smi rc:', nvidia_output.returncode)
    print(nvidia_output.stdout)
    print(nvidia_output.stderr)
except Exception as error:
    print('nvidia-smi error:', repr(error))
device_nodes = sorted(str(path) for path in Path('/dev').glob('nvidia*'))
print('nvidia device nodes:', device_nodes)
if not device_nodes and shutil.which('nvidia-smi') is None:
    raise RuntimeError(
        'No NVIDIA device was attached to this session; refusing CPU fallback'
    )

wheel_candidates = sorted(Path('/kaggle/input').rglob('torch-*cu128*.whl'))
if not wheel_candidates:
    raise FileNotFoundError('The offline CUDA PyTorch wheel was not mounted')
torch_wheel = wheel_candidates[0]
print('torch wheel:', torch_wheel)
valid_name = torch_wheel.name.replace('2.10.0cu128', '2.10.0+cu128')
valid_wheel = repo.parent / valid_name
shutil.copy2(torch_wheel, valid_wheel)
print('normalized wheel:', valid_wheel)

nvidia_wheels = sorted(
    path
    for path in Path('/kaggle/input').rglob('nvidia_*.whl')
    if 'cu12' in path.name and '-py3-none-' in path.name
)
if not nvidia_wheels:
    raise FileNotFoundError(
        'Offline CUDA 12 NVIDIA dependency wheels were not mounted'
    )
print('nvidia wheels:', len(nvidia_wheels))
OFFICIAL_WHEEL_HASHES = {'nvidia_cublas_cu12-12.8.4.1-py3-none-manylinux_2_27_aarch64.whl': 'b86f6dd8935884615a0683b663891d43781b819ac4f2ba2b0c9604676af346d0', 'nvidia_cublas_cu12-12.8.4.1-py3-none-manylinux_2_27_x86_64.whl': '8ac4e771d5a348c551b2a426eda6193c19aa630236b418086020df5ba9667142', 'nvidia_cublas_cu12-12.8.4.1-py3-none-win_amd64.whl': '47e9b82132fa8d2b4944e708049229601448aaad7e6f296f630f2d1a32de35af', 'nvidia_cuda_cupti_cu12-12.8.90-py3-none-manylinux2014_aarch64.manylinux_2_17_aarch64.whl': '4412396548808ddfed3f17a467b104ba7751e6b58678a4b840675c56d21cf7ed', 'nvidia_cuda_cupti_cu12-12.8.90-py3-none-manylinux2014_x86_64.manylinux_2_17_x86_64.whl': 'ea0cb07ebda26bb9b29ba82cda34849e73c166c18162d3913575b0c9db9a6182', 'nvidia_cuda_cupti_cu12-12.8.90-py3-none-win_amd64.whl': 'bb479dcdf7e6d4f8b0b01b115260399bf34154a1a2e9fe11c85c517d87efd98e', 'nvidia_cuda_nvrtc_cu12-12.8.93-py3-none-manylinux2010_x86_64.manylinux_2_12_x86_64.whl': 'a7756528852ef889772a84c6cd89d41dfa74667e24cca16bb31f8f061e3e9994', 'nvidia_cuda_nvrtc_cu12-12.8.93-py3-none-manylinux2014_aarch64.manylinux_2_17_aarch64.whl': 'fc1fec1e1637854b4c0a65fb9a8346b51dd9ee69e61ebaccc82058441f15bce8', 'nvidia_cuda_nvrtc_cu12-12.8.93-py3-none-win_amd64.whl': '7a4b6b2904850fe78e0bd179c4b655c404d4bb799ef03ddc60804247099ae909', 'nvidia_cuda_runtime_cu12-12.8.90-py3-none-manylinux2014_aarch64.manylinux_2_17_aarch64.whl': '52bf7bbee900262ffefe5e9d5a2a69a30d97e2bc5bb6cc866688caa976966e3d', 'nvidia_cuda_runtime_cu12-12.8.90-py3-none-manylinux2014_x86_64.manylinux_2_17_x86_64.whl': 'adade8dcbd0edf427b7204d480d6066d33902cab2a4707dcfc48a2d0fd44ab90', 'nvidia_cuda_runtime_cu12-12.8.90-py3-none-win_amd64.whl': 'c0c6027f01505bfed6c3b21ec546f69c687689aad5f1a377554bc6ca4aa993a8', 'nvidia_cudnn_cu12-9.10.2.21-py3-none-manylinux_2_27_aarch64.whl': 'c9132cc3f8958447b4910a1720036d9eff5928cc3179b0a51fb6d167c6cc87d8', 'nvidia_cudnn_cu12-9.10.2.21-py3-none-manylinux_2_27_x86_64.whl': '949452be657fa16687d0930933f032835951ef0892b37d2d53824d1a84dc97a8', 'nvidia_cudnn_cu12-9.10.2.21-py3-none-win_amd64.whl': 'c6288de7d63e6cf62988f0923f96dc339cea362decb1bf5b3141883392a7d65e', 'nvidia_cufft_cu12-11.3.3.83-py3-none-manylinux2014_aarch64.manylinux_2_17_aarch64.whl': '848ef7224d6305cdb2a4df928759dca7b1201874787083b6e7550dd6765ce69a', 'nvidia_cufft_cu12-11.3.3.83-py3-none-manylinux2014_x86_64.manylinux_2_17_x86_64.whl': '4d2dd21ec0b88cf61b62e6b43564355e5222e4a3fb394cac0db101f2dd0d4f74', 'nvidia_cufft_cu12-11.3.3.83-py3-none-win_amd64.whl': '7a64a98ef2a7c47f905aaf8931b69a3a43f27c55530c698bb2ed7c75c0b42cb7', 'nvidia_cufile_cu12-1.13.1.3-py3-none-manylinux2014_x86_64.manylinux_2_17_x86_64.whl': '1d069003be650e131b21c932ec3d8969c1715379251f8d23a1860554b1cb24fc', 'nvidia_cufile_cu12-1.13.1.3-py3-none-manylinux_2_27_aarch64.whl': '4beb6d4cce47c1a0f1013d72e02b0994730359e17801d395bdcbf20cfb3bb00a', 'nvidia_curand_cu12-10.3.9.90-py3-none-manylinux_2_27_aarch64.whl': 'dfab99248034673b779bc6decafdc3404a8a6f502462201f2f31f11354204acd', 'nvidia_curand_cu12-10.3.9.90-py3-none-manylinux_2_27_x86_64.whl': 'b32331d4f4df5d6eefa0554c565b626c7216f87a06a4f56fab27c3b68a830ec9', 'nvidia_curand_cu12-10.3.9.90-py3-none-win_amd64.whl': 'f149a8ca457277da854f89cf282d6ef43176861926c7ac85b2a0fbd237c587ec', 'nvidia_cusolver_cu12-11.7.3.90-py3-none-manylinux_2_27_aarch64.whl': 'db9ed69dbef9715071232caa9b69c52ac7de3a95773c2db65bdba85916e4e5c0', 'nvidia_cusolver_cu12-11.7.3.90-py3-none-manylinux_2_27_x86_64.whl': '4376c11ad263152bd50ea295c05370360776f8c3427b30991df774f9fb26c450', 'nvidia_cusolver_cu12-11.7.3.90-py3-none-win_amd64.whl': '4a550db115fcabc4d495eb7d39ac8b58d4ab5d8e63274d3754df1c0ad6a22d34', 'nvidia_cusparse_cu12-12.5.8.93-py3-none-manylinux2014_aarch64.manylinux_2_17_aarch64.whl': '9b6c161cb130be1a07a27ea6923df8141f3c295852f4b260c65f18f3e0a091dc', 'nvidia_cusparse_cu12-12.5.8.93-py3-none-manylinux2014_x86_64.manylinux_2_17_x86_64.whl': '1ec05d76bbbd8b61b06a80e1eaf8cf4959c3d4ce8e711b65ebd0443bb0ebb13b', 'nvidia_cusparse_cu12-12.5.8.93-py3-none-win_amd64.whl': '9a33604331cb2cac199f2e7f5104dfbb8a5a898c367a53dfda9ff2acb6b6b4dd', 'nvidia_cusparselt_cu12-0.7.1-py3-none-manylinux2014_aarch64.whl': '8878dce784d0fac90131b6817b607e803c36e629ba34dc5b433471382196b6a5', 'nvidia_cusparselt_cu12-0.7.1-py3-none-manylinux2014_x86_64.whl': 'f1bb701d6b930d5a7cea44c19ceb973311500847f81b634d802b7b539dc55623', 'nvidia_cusparselt_cu12-0.7.1-py3-none-win_amd64.whl': 'f67fbb5831940ec829c9117b7f33807db9f9678dc2a617fbe781cac17b4e1075', 'nvidia_nccl_cu12-2.27.5-py3-none-manylinux2014_aarch64.manylinux_2_17_aarch64.whl': '31432ad4d1fb1004eb0c56203dc9bc2178a1ba69d1d9e02d64a6938ab5e40e7a', 'nvidia_nccl_cu12-2.27.5-py3-none-manylinux2014_x86_64.manylinux_2_17_x86_64.whl': 'ad730cf15cb5d25fe849c6e6ca9eb5b76db16a80f13f425ac68d8e2e55624457', 'nvidia_nvjitlink_cu12-12.8.93-py3-none-manylinux2010_x86_64.manylinux_2_12_x86_64.whl': '81ff63371a7ebd6e6451970684f916be2eab07321b73c9d244dc2b4da7f73b88', 'nvidia_nvjitlink_cu12-12.8.93-py3-none-manylinux2014_aarch64.manylinux_2_17_aarch64.whl': 'adccd7161ace7261e01bb91e44e88da350895c270d23f744f0820c818b7229e7', 'nvidia_nvjitlink_cu12-12.8.93-py3-none-win_amd64.whl': 'bd93fbeeee850917903583587f4fc3a4eafa022e34572251368238ab5e6bd67f', 'nvidia_nvshmem_cu12-3.4.5-py3-none-manylinux2014_aarch64.manylinux_2_17_aarch64.whl': '0b48363fc6964dede448029434c6abed6c5e37f823cb43c3bcde7ecfc0457e15', 'nvidia_nvshmem_cu12-3.4.5-py3-none-manylinux2014_x86_64.manylinux_2_17_x86_64.whl': '042f2500f24c021db8a06c5eec2539027d57460e1c1a762055a6554f72c369bd', 'nvidia_nvtx_cu12-12.8.90-py3-none-manylinux2014_aarch64.manylinux_2_17_aarch64.whl': 'd7ad891da111ebafbf7e015d34879f7112832fc239ff0d7d776b6cb685274615', 'nvidia_nvtx_cu12-12.8.90-py3-none-manylinux2014_x86_64.manylinux_2_17_x86_64.whl': '5b17e2001cc0d751a5bc2c6ec6d26ad95913324a4adb86788c944f8ce9ba441f', 'nvidia_nvtx_cu12-12.8.90-py3-none-win_amd64.whl': '619c8304aedc69f02ea82dd244541a83c3d9d40993381b3b590f1adaed3db41e', 'torch-2.10.0+cu128-cp313-cp313-manylinux_2_28_x86_64.whl': '7b4bd23ed63de97456fcc81c26fea9f02ee02ce1112111c4dac0d8cfe574b23e'}

import hashlib

def wheel_sha256(path):
    digest = hashlib.sha256()
    with path.open('rb') as stream:
        for block in iter(lambda: stream.read(8 * 1024 * 1024), b''):
            digest.update(block)
    return digest.hexdigest()

vendor_hash = OFFICIAL_WHEEL_HASHES.get(valid_name)
if not vendor_hash or wheel_sha256(torch_wheel) != vendor_hash:
    raise RuntimeError('Offline torch wheel does not match the official PyTorch index')
print('Official PyTorch SHA256 verified:', valid_name, vendor_hash, flush=True)

for path in nvidia_wheels:
    expected_hash = OFFICIAL_WHEEL_HASHES.get(path.name)
    if not expected_hash or wheel_sha256(path) != expected_hash:
        raise RuntimeError('Offline NVIDIA wheel does not match official PyPI: ' + path.name)
    print('Official PyPI SHA256 verified:', path.name, flush=True)

subprocess.run(
    [
        sys.executable, '-m', 'pip', 'install', '--no-deps',
        *[str(path) for path in nvidia_wheels],
    ],
    check=True,
)
subprocess.run(
    [
        sys.executable, '-m', 'pip', 'install', '--no-deps',
        '--force-reinstall', str(valid_wheel),
    ],
    check=True,
)
subprocess.run(
    [
        sys.executable, '-c',
        'import torch; print("installed torch:", torch.__version__); '
        'print("installed cuda:", torch.version.cuda); '
        'print("installed cuda available:", torch.cuda.is_available())',
    ],
    check=True,
)
print('repository:', repo)
print('dataset:', data_root)
print('config:', config_path)
print('python:', sys.version)
subprocess.run([
    sys.executable, '-c',
    "import torch; assert torch.__version__ == '2.10.0+cu128', torch.__version__; "
    "assert torch.cuda.is_available(), 'CUDA unavailable'; "
    "print('Baseline torch 2.10.0+cu128 + CUDA: PASS')"
], check=True)
valid_wheel.unlink(missing_ok=True)

In [ ]:
import hashlib
import json
import textwrap

E3_TESTS_PASSED = False
expected_split_sha256 = '4853e4fe55dac627ae4a5daba918aa00f46feb156932e909cbce5802742a7052'
actual_hash = hashlib.sha256((repo / 'data/splits/main_split.json').read_bytes()).hexdigest()
assert actual_hash == expected_split_sha256, 'Frozen split checksum mismatch'

def run_checked(arguments, log_name):
    result = subprocess.run(
        [sys.executable, *arguments], cwd=repo, env=os.environ.copy(),
        stdout=subprocess.PIPE, stderr=subprocess.STDOUT, text=True,
    )
    (repo / log_name).write_text(result.stdout, encoding='utf-8')
    print(result.stdout)
    result.check_returncode()

run_checked(['-m', 'unittest', 'discover', '-s', 'tests', '-v'], 'e3_tests.log')

smoke = textwrap.dedent("""
    import torch
    import yaml
    from src.losses import FocalTverskyLoss, build_loss
    from src.models import build_model
    from src.utils import seed_everything

    with open('configs/experiments/E3_focal_tversky.yaml', encoding='utf-8') as f:
        config = yaml.safe_load(f)
    with open('configs/baseline.yaml', encoding='utf-8') as f:
        baseline = yaml.safe_load(f)
    for key in ('model', 'train', 'evaluation'):
        assert config[key] == baseline[key], key
    for key, value in baseline['data'].items():
        if key != 'generate_split_if_missing':
            assert config['data'][key] == value, key
    assert config['project']['seed'] == baseline['project']['seed'] == 42
    assert config['data']['generate_split_if_missing'] is False
    assert config['project']['output_dir'] != baseline['project']['output_dir']
    assert torch.cuda.is_available(), 'CUDA unavailable; stop before training'
    seed_everything(42)
    model = build_model(config['model']).cuda()
    criterion = build_loss(config['loss'])
    assert isinstance(criterion, FocalTverskyLoss), 'Wrong loss selected'
    image = torch.randn(1, 1, 256, 256, device='cuda')
    mask = torch.zeros_like(image)
    mask[:, :, 100:110, 100:110] = 1
    optimizer = torch.optim.AdamW(model.parameters(), lr=0.0003)
    scaler = torch.amp.GradScaler('cuda')
    with torch.autocast(device_type='cuda', enabled=True):
        loss = criterion(model(image), mask)
    assert torch.isfinite(loss)
    scaler.scale(loss).backward()
    scaler.unscale_(optimizer)
    assert all(torch.isfinite(p.grad).all() for p in model.parameters() if p.grad is not None)
    scaler.step(optimizer)
    scaler.update()
    print('E3 factory + CUDA AMP backward: PASS; loss=', float(loss.detach()))
""")
run_checked(['-c', smoke], 'e3_cuda_smoke.log')
E3_TESTS_PASSED = True
print('验证通过。尚未正式训练；确认结果后再启用下一格。')

In [ ]:
# 默认关闭：先检查上一格的测试结果，再决定是否启动完整训练。
RUN_TRAINING = False
if RUN_TRAINING:
    assert E3_TESTS_PASSED, '先完成测试及 CUDA 验证'
    subprocess.run([
        sys.executable, '-c',
        "import json, platform, torch; from pathlib import Path; "
        "p=Path('reports/experiments/E3_focal_tversky'); p.mkdir(parents=True, exist_ok=True); "
        "(p/'environment.json').write_text(json.dumps({'python':platform.python_version(), "
        "'torch':torch.__version__, 'cuda':torch.version.cuda, "
        "'cudnn':torch.backends.cudnn.version(), 'gpu':torch.cuda.get_device_name(0)}, "
        "indent=2), encoding='utf-8')"
    ], cwd=repo, check=True, env=os.environ.copy())
    subprocess.run([sys.executable, 'scripts/train.py', '--config', config_path],
                   cwd=repo, check=True, env=os.environ.copy())
else:
    print('正式训练未启动。')

In [ ]:
# 仅在验证集选定配置和最佳权重后开启；不要用测试集或 External_Test 调参。
RUN_FINAL_EVALUATION = False
if RUN_FINAL_EVALUATION:
    assert E3_TESTS_PASSED, '先完成测试'
    assert (repo / 'reports/experiments/E3_focal_tversky/best.pt').is_file()
    for entry in ('evaluate.py', 'infer.py'):
        subprocess.run([sys.executable, 'scripts/' + entry, '--config', config_path],
                       cwd=repo, check=True, env=os.environ.copy())
else:
    print('最终测试与外部评估未启动。')

In [ ]:
# 打包当前实验产出（可能只有验证日志），供下载保存。
import zipfile
bundle = Path('/kaggle/working') / (repo.name + '-results.zip')
with zipfile.ZipFile(bundle, 'w', compression=zipfile.ZIP_DEFLATED) as archive:
    for path in repo.rglob('*'):
        if path.is_file() and (
            'E3_focal_tversky' in path.parts or path.name in ('e3_tests.log', 'e3_cuda_smoke.log')
        ):
            archive.write(path, path.relative_to(repo))
print('结果包：', bundle)
print('本格不会启动训练或评估。权重可保留在下载包中，但不要提交到 Git。')